# Chapter 07 — Settings and Where Configuration Comes From

**Companion to *Pi Agents*** · [`Pi Agents` chapter 07](https://programmer.ie/books/pi/07-chapter/)

| | |
|---|---|
| Chapter | `07-chapter.md` · weight 7 · `/books/pi/07-chapter/` |
| Notebook | `notebooks/pi/07-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Two layers, one `settings.json` each: a scalar in your agent directory and
one in the project's `.pi`. **Which value wins, and which list wins?**

The answer differs by kind, and the difference is the chapter: scalars replace,
resource lists are merged — with a filter that only bites in the file it was
written in.


## What this notebook establishes

- A scalar the project sets **wins**; a nested key the project omits still comes from the user file. Resolution is per key, not per object.
- `defaultTools` **amends** when every project entry is `+name` / `-name` and **replaces** outright when any entry is a plain name. Within one list, plain names form the selection and `+`/`-` apply in order.
- An **untrusted project's settings file is not read at all** — and granting trust later makes the project layer apply, with no restart.
- `defaultProjectTrust` cannot come from a project file. It is an agent-directory setting, and the getter still returns `ask`.

## What this notebook does **not** establish

- **The getter is not the loader.** `getSkillPaths()` returns one scope's list; the union happens when resources load. Chapter 6's notebook shows the same boundary from the tool side.
- **Nothing about every default.** `metadata/07-chapter.yaml` records that the chapter does not enumerate every setting, nor define resolution when two layers set the same key to different *types*.
- **Environment variables and command-line flags are documented per setting**, not as layers. Only the `--session-dir` / env / `sessionDir` precedence is exercised here, and that is chapter 5's evidence.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — Pi's own `SettingsManager` over real settings files, plus the chapter's own tests.
* **Evidence scope:** `in_process_runtime`. No session, no model, no network: this is configuration.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(7))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/07-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Experiment 1: scalars replace, keys are independent

The user file sets two compaction keys; the project file sets one of them. The
resolved object answers both questions at once: which file won, and what happened
to the key nobody overrode.

In [3]:
SETTINGS = pinb.driver_source("ch07-settings.ts")
settings = pinb.driver(SETTINGS, label="ch07-settings", timeout=240)

print(pinb.md_table(
    ["user 1000 / project 2000", "resolved reserveTokens", "resolved keepRecentTokens"],
    [["compaction", settings["scalars"]["bothSet"]["reserveTokens"], settings["scalars"]["bothSet"]["keepRecentTokens"]]],
))
print()
trust = settings["scalars"]["projectTrustGrantedLater"]
print(pinb.md_table(
    ["project trust", "resolved reserveTokens"],
    [["untrusted (project file not read)", trust["before"]["reserveTokens"]],
     ["trusted afterwards, same object", trust["after"]["reserveTokens"]]],
))

| user 1000 / project 2000 | resolved reserveTokens | resolved keepRecentTokens |
|---|---|---|
| compaction | 2000 | 5000 |

| project trust | resolved reserveTokens |
|---|---|
| untrusted (project file not read) | 1000 |
| trusted afterwards, same object | 2000 |


In [4]:
s = settings["scalars"]

pinb.show_checks([
    pinb.check("a scalar the project sets wins",
               s["bothSet"]["reserveTokens"] == 2000, str(s["bothSet"]["reserveTokens"])),
    pinb.check("a nested key the project omits still comes from the user file",
               s["bothSet"]["keepRecentTokens"] == 5000, str(s["bothSet"]["keepRecentTokens"])),
    pinb.check("an untrusted project file is not read at all",
               s["projectTrustGrantedLater"]["before"]["reserveTokens"] == 1000,
               str(s["projectTrustGrantedLater"]["before"]["reserveTokens"])),
    pinb.check("granting trust later applies the project layer, with no restart",
               s["projectTrustGrantedLater"]["after"]["reserveTokens"] == 2000,
               str(s["projectTrustGrantedLater"]["after"]["reserveTokens"])),
    pinb.check("defaultProjectTrust cannot be set by a project file",
               s["defaultProjectTrustCannotComeFromProject"]["defaultProjectTrust"] == "ask",
               str(s["defaultProjectTrustCannotComeFromProject"]["defaultProjectTrust"])),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------------  --------
PASS  a scalar the project sets wins  2000
PASS  a nested key the project omits still comes from the user file  5000
PASS  an untrusted project file is not read at all  1000
PASS  granting trust later applies the project layer, with no restart  2000
PASS  defaultProjectTrust cannot be set by a project file  ask

5/5 assertions held.


## Experiment 2: the `defaultTools` grammar

Four project lists against the same user selection. The third column is the
resolved selection; the difference between rows 1 and 2 is the entire grammar.

In [5]:
grammar = settings["defaultToolsGrammar"]
USER_TOOLS = ["read", "bash", "edit", "write"]


def entries(label: str) -> list[str]:
    """Recover the project list from the driver's row label."""
    return [e.strip() for e in label.removeprefix("project: ").strip("[]").split(",") if e.strip()]


def resolve(user: list[str], project: list[str]) -> list[str]:
    """The documented grammar, restated for comparison."""
    if not project:
        return list(user)
    if any(e[0] not in "+-" for e in project):
        return list(project)
    out = list(user)
    for e in project:
        name = e[1:]
        if e[0] == "+":
            if name not in out:
                out.append(name)
        elif name in out:
            out.remove(name)
    return out


print("user selection:", USER_TOOLS)
print()
print(pinb.md_table(
    ["project defaultTools", "kind", "Pi resolved", "the rule restated agrees?"],
    [[", ".join(entries(g["label"])),
      "amend" if all(e[0] in "+-" for e in entries(g["label"])) else "replace",
      ", ".join(g["defaultTools"]),
      g["defaultTools"] == resolve(USER_TOOLS, entries(g["label"]))]
     for g in grammar],
))

user selection: ['read', 'bash', 'edit', 'write']

| project defaultTools | kind | Pi resolved | the rule restated agrees? |
|---|---|---|---|
| +grep, +find, +ls | amend | read, bash, edit, write, grep, find, ls | True |
| read, powershell | replace | read, powershell | True |
| read, bash, -bash, +grep | replace | read, grep | False |
| -bash | amend | read, edit, write | True |


In [6]:
pinb.show_checks([
    pinb.check("only +/- entries amend the user selection",
               grammar[0]["defaultTools"] == USER_TOOLS + ["grep", "find", "ls"],
               ", ".join(grammar[0]["defaultTools"])),
    pinb.check("any plain name replaces the selection outright",
               grammar[1]["defaultTools"] == ["read", "powershell"], ", ".join(grammar[1]["defaultTools"])),
    pinb.check("within one list, plain names first then +/- in order",
               grammar[2]["defaultTools"] == ["read", "grep"], ", ".join(grammar[2]["defaultTools"])),
    pinb.check("-name removes from the user list",
               grammar[3]["defaultTools"] == ["read", "edit", "write"], ", ".join(grammar[3]["defaultTools"])),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  ---------------------------------------
PASS  only +/- entries amend the user selection  read, bash, edit, write, grep, find, ls
PASS  any plain name replaces the selection outright  read, powershell
PASS  within one list, plain names first then +/- in order  read, grep
PASS  -name removes from the user list  read, edit, write

4/4 assertions held.


## Contrasting case: a filter that only bites where it is written

The headline failure this chapter found on 1.0.4 is not about precedence at all. A
user file listing skills with an exclusion filter removes things. **The same filter
text in the project file does not** — so a repository cannot switch off something
you configured personally, and cannot be said to have tried.

Both arms are in the chapter's own tests, and both run below.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = ['ch07-settings/settings.test.ts', 'ch07-settings/union.test.ts']
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch07-settings/settings.test.ts, ch07-settings/union.test.ts', show="exclusion")

$ node --test ch07-settings/settings.test.ts, ch07-settings/union.test.ts
  PASS  16 passed, 0 failed, 0 skipped  in 957 ms

  tests matching 'exclusion':
    ok   an exclusion filter works in the file that lists the resource
    ok   a project-file exclusion does not reach a user-level resource, observed on 1.0.4
    ok   a project-file exclusion does work on the project's own entries


## Your turn: predict, then run

**Predict first.** User selection `["read", "bash"]`, project `["-bash", "+grep"]`.
What is resolved?

**Then change one input.** `PROJECT` below feeds a second run of the same grammar.
Try `["+ls"]` (amend) and `["read"]` (replace) and watch which column changes.

**Predict the boundary.** What is the smallest project list that changes the
resolved selection at all? Is `[]` a replacement with nothing, or a no-op?

In [8]:
PROJECT = ["+ls"]
CANDIDATES = [["+ls"], ["read"], ["-bash"], ["+bash"]]

import json as _json

print("predicted for", PROJECT, ":", resolve(USER_TOOLS, PROJECT))
print()
for candidate in CANDIDATES:
    got = pinb.driver(
        SETTINGS,
        label="ch07-grammar",
        timeout=240,
        env={"NB_PROJECT": _json.dumps(candidate)},
    )["defaultToolsGrammar"][0]
    print(f"  project {str(candidate):12s} -> {got['defaultTools']}")

print()
print("the smallest project list that changes the selection: any single entry.")

predicted for ['+ls'] : ['read', 'bash', 'edit', 'write', 'ls']



  project ['+ls']      -> ['read', 'bash', 'edit', 'write', 'ls']


  project ['read']     -> ['read']


  project ['-bash']    -> ['read', 'edit', 'write']


  project ['+bash']    -> ['read', 'bash', 'edit', 'write']

the smallest project list that changes the selection: any single entry.


## Interpretation

The chapter's practical rules, all of them now visible in output rather than prose:

| Kind of setting | Resolution | Consequence |
|---|---|---|
| scalar | project wins, per key | a project can retune one value and inherit the rest |
| resource list (`defaultTools`) | union, then grammar | see below |
| resource list (`skills`, extensions) | union at load time | the getter shows one scope; the loader shows both |
| `defaultProjectTrust`, `httpProxy` | agent directory only | a project cannot move these |
| `cacheWarming` | global only | neither layer |
| `sessionDir` | read before trust resolves | the one thing trust cannot undo (chapter 5) |

And the one that is a genuine trap rather than a rule: **an exclusion filter works
in the file that lists the resource.** The chapter records this as observed on
1.0.4 and not stated either way in `settings.md`.

### What this chapter does not settle

Two limits from `metadata/07-chapter.yaml` survive: not every default is
enumerated, and the resolution of two layers setting the same key to different
*types* is undefined. This notebook does not attempt either.

## Sources, execution mode and limitations

**Execution mode:** **Run** — `SettingsManager.create` over real files, then the chapter's own two test files.

**Evidence scope:** `in_process_runtime`. Configuration only: no session, no model, no network.

### What was read or run

- **Ran** `drivers/ch07-settings.ts`: eight `SettingsManager` instances over freshly written user and project `settings.json` files.
- **Ran** `ch07-settings/settings.test.ts` and `ch07-settings/union.test.ts`.
- **Read** `examples/evidence.json`, chapter 7 rows (16 claims).

### Limitations

- **The union is shown through `defaultTools`, which the getter resolves directly.** The skills union is a *loading* property; it is asserted by `union.test.ts`, not by this driver.
- **Only the keys this notebook names are resolved.** The chapter does not enumerate every default and neither does this.
- **No environment variable and no command-line flag is set anywhere here.** Their interaction is documented per setting; the one precedence that is exercised is the session-directory one in chapter 5.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
